In [4]:
import pandas as pd
import os

In [5]:
# Load lineage dataframe for Monastrol analysis

root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
outputFolder = root + "DataFrames" + "/"

df = pd.read_csv('/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/MainDataFrame_Lineages_Monastrol.csv')

In [6]:
# Parse mitotic error tables (Fiji semi-manual analysis)

error_path = [root + "Errors_Monastrol.xlsx"]

def parse_error(error_list = error_path):
    dfs = []
    for path in error_list:
        df = pd.read_excel(path)
        dfs.append(df)
    return pd.concat(dfs)

df_error = parse_error()
df_error = df_error.replace({"T": True, "F": False, "R": False})

# Fix dtypes in evaluation columns 
# (Mitotic death exluded from this list!)

cols = [
    "Laggings", 
    "Massive Missegregation", 
    "DNA bridge", 
    "Misaligned", 
    "Multipolar Divition", 
    'Citokinesis Failure',
    'Slippage',
    'MN' 
       ]

# Work on a copy to avoid warnings
bool_df = df_error.loc[:, cols].copy()

# Convert and cast to nullable boolean
bool_df = bool_df.apply(lambda col: col.astype("boolean"))

# Convert values: keep empty as NaN, only "True"/True as True, "False"/False as False
# TODO try to get this to work with the original df 
def to_bool_or_nan(x):
    if x in [True, "True"]:
        return True
    elif x in [False, "False"]:
        return False
    else:
        return np.nan   # preserve empties

bool_df = bool_df.map(to_bool_or_nan)
bool_df.loc[:, "any_true"] = bool_df.any(axis = 1, bool_only = True)

# New column: True if any True, NaN if all are NaN, otherwise False
df_error.loc[:, "any_true"] = bool_df.any(axis = 1)

print(df_error.groupby("Dataset")["any_true"].value_counts(dropna = False))

df_error.head()

Dataset     any_true
20250627.0  False       123
            True          7
20250704.0  False        73
            True         15
Name: count, dtype: int64


/var/folders/lr/cb1nd6l97696j8chvkss86580000gn/T/ipykernel_33344/2032306735.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_error = df_error.replace({"T": True, "F": False, "R": False})


,Source_ID,Dataset,Position,Laggings,Massive Missegregation,DNA bridge,Misaligned,MN,Multipolar Divition,Citokinesis Failure,Slippage,Mitotic Death,Comments,any_true
0,2547.0,20250627.0,3.0,False,False,False,False,False,False,False,False,False,where is mitosis,False
1,2574.0,20250627.0,3.0,False,False,False,False,False,False,False,False,False,NaN,False
2,5610.0,20250627.0,3.0,False,False,False,False,False,False,False,False,False,NaN,False
3,6378.0,20250627.0,3.0,False,False,False,False,False,False,False,False,False,NaN,False
4,2884.0,20250627.0,3.0,False,False,False,False,False,False,False,False,False,NaN,False


In [7]:
def getonsets(filepath, position, dataset):
    onsets = pd.read_csv(filepath)
    onsets["Source_ID"] = onsets.Label.str.split(":").str.get(1).astype(int)
    onsets["Position"] = position
    onsets["Dataset"] = int(dataset)
    #onsets["Dataset"] = onsets.Dataset.astype(int)
    onsets.rename(columns = {"Slice": "Frame_onset"}, inplace = True)
    return onsets

# Define metadata for each file
onsets_info = [
    {"date": "20250627", "position": 3, "file": "20250627_IM_H2B-GFP_MitoStop_02_onsets.csv"},
    {"date": "20250627", "position": 4, "file": "20250627_IM_H2B-GFP_MitoStop_02_onsets.csv"},
    {"date": "20250704", "position": 1, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 2, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 3, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 4, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 5, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 6, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 7, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
]

# Generate and combine dataframes
df_onsets_list = []

for info in onsets_info:
    folder = f"{root}/{info['date']}/Position_{info['position']}_MonastrolWashout/Concatenated"
    filepath = os.path.join(folder, info["file"])
    df_single = getonsets(filepath, info["position"], info["date"])
    df_onsets_list.append(df_single)

df_onsets = pd.concat(df_onsets_list, ignore_index = True)

In [8]:
df["Frame"] = df.Frame.astype(int) + 1 # TODO CHECK ALIGNMENT

df = df.merge(df_onsets, how = "outer")

,Unnamed: 0,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,Position,Dataset,...,Cell_Cycle_mins,Cell_Cycle_hours,Seen_sister,Random_ID,Seen_granny,,Label,X,Y,Frame_onset
0,0,0,2011,True,2011,444.566899,406.863649,66,3,20250627,...,NaN,NaN,NaN,2925,NaN,22,tomeasure:2011:11,171.529,747.022,11
1,0,0,2012,True,2012,761.937434,648.137100,59,3,20250704,...,NaN,NaN,NaN,2671,NaN,52,tomeasure:2012:14,819.528,643.027,14
2,2,0,2046,True,2012.2046,739.287853,889.272274,461,3,20250704,...,2814.0,46.900000,NaN,5531,NaN,22,tomeasure:2046:444,742.464,919.794,444
3,4,1,2081,True,2081,580.740785,683.768758,58,3,20250704,...,NaN,NaN,NaN,5091,NaN,46,tomeasure:2081:48,568.449,727.549,48
4,2,0,2105,True,2011.2105,452.646167,440.838007,527,3,20250627,...,3227.0,53.783333,True,5905,NaN,76,tomeasure:2105:500,454.096,473.569,500


In [9]:
def getInterphaseDuration(x, dataframe = df):
    daughter_time = x.Frame_onset
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        if mother_row["Frame"].shape[0] == 1:   
            mother_time = mother_row["Frame"].item()
            interphase_duration = (int(daughter_time) - int(mother_time)) * 7
            return interphase_duration
        else:
            return None


def getMitoticDurationMother(x, dataframe = df):
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        
        if len(mother_row) == 1:
            return mother_row.iloc[0].Mitotic_duration_mins
        else:
            return None


def getMetaphaseArrest(x):
    # mitotic duration bins, from mother perspective
    if x < 30:
        return "000-030 min"
    elif (x < 60) & (x > 30):
        return "030-060"
    elif (x < 90) & (x > 60):
        return "060-090"
    elif (x < 120) & (x > 90):
        return "090-120"
    else:
        return "> 120 min"


def getMetaphaseArrest_mother(x):
    # mitotic duration bins, from daughter perspective
    if x > 0:
        if x < 30:
            return "000-030 min"
        elif (x < 60) & (x > 30):
            return "030-060"
        elif (x < 90) & (x > 60):
            return "060-090"
        elif (x < 120) & (x > 90):
            return "090-120"
        else:
            return "> 120 min"
    else:
        pass

df["Frames_in_mitosis"] = (df.Frame.astype(int) - df.Frame_onset.astype(int))
df["Mitotic_duration_mins"] = df.Frames_in_mitosis * 7

df["Interphase_duration_mins"] = df.apply(getInterphaseDuration, axis = 1) 
df["Interphase_duration_hrs"] = df.Interphase_duration_mins / 60
df["Mitotic_duration_hrs"] = df.Mitotic_duration_mins / 60

df["Mother_Mitotic_duration_mins"] = df.apply(getMitoticDurationMother, axis = 1)
df["Mother_arrested"] = df.Mother_Mitotic_duration_mins.apply(getMetaphaseArrest_mother)
df["Metaphase_arrested"] = df.Mitotic_duration_mins.apply(getMetaphaseArrest)
df["Frame_onset_hrs"] = df.Frame_onset * 7 / 60 
df["Frame_hrs"] = df.Frame * 7 / 60 

# Create Bins (one bin for every 30 min of mitotic duration) # kind of the same as "Metaphase_arrested"
interval_range_mitosis = pd.interval_range(start = 0, freq = 30, end = 900) 
df['Mitotic_duration_bin'] = pd.cut(df['Mitotic_duration_mins'], bins = interval_range_mitosis).astype(str)

df["Unique_Track_ID"] = df.Dataset.astype(str) + df.Position.astype(str) + df.Track_ID.astype(str)
df["Unique_Source_ID"] = df.Dataset.astype(str) + df.Position.astype(str) + df.Source_ID.astype(str)

# Drop erronous annotations / trackings
df = df.drop(df[df.Cell_Cycle_hours < 1].index)
df = df.drop(df[df.Mitotic_duration_mins < 0].index)
df = df.drop(df[df.Mother_Mitotic_duration_mins < 0].index)

,Unnamed: 0,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,Position,Dataset,...,Interphase_duration_hrs,Mitotic_duration_hrs,Mother_Mitotic_duration_mins,Mother_arrested,Metaphase_arrested,Frame_onset_hrs,Frame_hrs,Mitotic_duration_bin,Unique_Track_ID,Unique_Source_ID
0,0,0,2011,True,2011,444.566899,406.863649,66,3,20250627,...,NaN,6.416667,NaN,None,> 120 min,1.283333,7.700000,"(360, 390]",2025062730,2025062732011
1,0,0,2012,True,2012,761.937434,648.137100,59,3,20250704,...,NaN,5.250000,NaN,None,> 120 min,1.633333,6.883333,"(300, 330]",2025070430,2025070432012
2,2,0,2046,True,2012.2046,739.287853,889.272274,461,3,20250704,...,44.916667,1.983333,315.0,> 120 min,090-120,51.800000,53.783333,"(90, 120]",2025070430,2025070432046
3,4,1,2081,True,2081,580.740785,683.768758,58,3,20250704,...,NaN,1.166667,NaN,None,060-090,5.600000,6.766667,"(60, 90]",2025070431,2025070432081
4,2,0,2105,True,2011.2105,452.646167,440.838007,527,3,20250627,...,50.633333,3.150000,385.0,> 120 min,> 120 min,58.333333,61.483333,"(180, 210]",2025062730,2025062732105


In [11]:
# Final (outer) merge with error dataframe
# Note: error annotations are until 18h imaging time (Frames < 155)

df = df.merge(df_error, how = "outer")

In [12]:
df.to_csv(outputFolder + "MainDataFrame_MitoticStopwatch_Monastrol.csv")

print("Finished.")

Finished.
